# 🕵️ Dia 53 — Forense de memória: aquisição e árvore de processos

**Investigador Cyber 90 Dias** · Fase 4 · Rank Jutsu `B`

> Baseado no *Day 53* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-4-digital-forensics-incident-investigation/day-53.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

A memória do `SA-CX-07` foi capturada às 17:41 UTC, antes de o notebook ser desligado. O `msa-atualizador.exe` é filho do `explorer.exe`. *"Pai normal, caminho estranho"*, diz Rafa. *"E cadê o `schtasks.exe`?"*

> ⚠️ **Honestidade do laboratório:** não há imagem de memória real, nem do WS-FIN-07 (original) nem do `SA-CX-07`. Os arquivos são texto no layout do Volatility 3. Para rodar o `vol` de verdade, use uma imagem do [MemLabs](https://github.com/stuxnet999/MemLabs).

Neste notebook você vai:

1. **Interpretar o `windows.pstree.txt`** e reconstruir a árvore.
2. **Transformar a árvore normal do Windows em regras** e listar desvios por coluna.
3. **Decodificar o `-EncodedCommand`** (e, no caso BR, o `--canal`) sem executar nada.
4. **Gerar o `pstree.dot`** com nós vermelhos e, se houver Graphviz, o PNG.
5. **Comparar `pslist` e `psscan`** do `SA-CX-07`.
6. **Jogar o quiz do pai certo.**


In [ ]:
# (OPCIONAL) Graphviz e Volatility 3 reais no Colab.
# !apt-get -qq install -y graphviz > /dev/null
# !pip -q install volatility3
# Com uma imagem do MemLabs (calcule o hash antes!):  !vol -f MemoryDump_Lab1.raw windows.pstree
print("Célula opcional.")


In [ ]:
# ------------------------------------------------------------
# RECEBER O EVID-003 (PSTREE E CMDLINE DO WS-FIN-07)
# Plano A: baixar do repositório original (GitHub raw), com urllib.
# Plano B: usar a cópia embutida logo abaixo (o arquivo inteiro,
#          copiado do upstream/), para o notebook rodar sem internet.
# ------------------------------------------------------------
import urllib.request      # baixa arquivos da web, sem instalar nada
import hashlib             # SHA-256: a impressão digital dos bytes

FORCAR_OFFLINE = False     # True = nem tenta a rede (útil em sala sem Wi-Fi)
URL_BASE = 'https://raw.githubusercontent.com/thiefdroppers/cyber-investigator-skills/main/curriculum/phase-4-digital-forensics-incident-investigation/resources/'

RESERVA = {}             # caminho do arquivo -> conteúdo embutido (plano B)
RESERVA['case-lab-p4/memory/windows.pstree.txt'] = ''.join((
    '# SYNTHETIC TRAINING DATA (case LAB-P4). Hand-written to resemble Volatility 3 windows.pstree\n'
    '# text output for a fictional host WS-FIN-07. No real memory image exists behind it. Columns\n'
    '# follow Volatility 3 2.x layout; exact columns vary by version. Tab-separated.\n'
    'PID\tPPID\tImageFileName\tOffset(V)\tThreads\tHandles\tSessionId\tWow64\tCreateTime\tExitTime\tAudit\tCmd\tPath\n'
    '4\t0\tSystem\t0xa3066c682040\t152\t-\tN/A\tFalse\t2026-03-13 12:58:01.000000 UTC\tN/A\t-\t-\t-\n'
    '* 92\t4\tRegistry\t0xa3066c6f1080\t4\t-\tN/A\tFalse\t2026-03-13 12:57:58.000000 UTC\tN/A\tRegistry\t-\t-\n'
    '* 348\t4\tsmss.exe\t0xa3066f0b4040\t2\t-\tN/A\tFalse\t2026-03-13 12:58:01.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\smss.exe\t\\SystemRoot\\System32\\smss.exe\t\\SystemRoot\\System32\\smss.exe\n'
    '456\t444\tcsrss.exe\t0xa3066f6a3140\t11\t-\t0\tFalse\t2026-03-13 12:58:03.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\csrss.exe\t%SystemRoot%\\system32\\csrss.exe ObjectDirectory=\\Windows SharedSection=1024,20480,768 Windows=On SubSystemType=Windows ServerDll=basesrv,1 ServerDll=winsrv:UserServerDllInitialization,3 ServerDll=sxssrv,4 ProfileControl=Off MaxRequestThreads=16\tC:\\Windows\\system32\\csrss.exe\n'
    '532\t444\twininit.exe\t0xa3066f6d8080\t1\t-\t0\tFalse\t2026-03-13 12:58:03.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\wininit.exe\twininit.exe\tC:\\Windows\\system32\\wininit.exe\n'
    '* 644\t532\tservices.exe\t0xa3066f73a0c0\t6\t-\t0\tFalse\t2026-03-13 12:58:03.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\services.exe\tC:\\Windows\\system32\\services.exe\tC:\\Windows\\system32\\services.exe\n'
    '** 800\t644\tsvchost.exe\t0xa3066f7d1080\t21\t-\t0\tFalse\t2026-03-13 12:58:04.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\svchost.exe\tC:\\Windows\\system32\\svchost.exe -k DcomLaunch -p\tC:\\Windows\\system32\\svchost.exe\n'
    '** 852\t644\tsvchost.exe\t0xa3066f7f3080\t12\t-\t0\tFalse\t2026-03-13 12:58:04.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\svchost.exe\tC:\\Windows\\system32\\svchost.exe -k RPCSS -p\tC:\\Windows\\system32\\svchost.exe\n'
    '** 1100\t644\tsvchost.exe\t0xa3066fa1e080\t38\t-\t0\tFalse\t2026-03-13 12:58:05.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\svchost.exe\tC:\\Windows\\system32\\svchost.exe -k netsvcs -p\tC:\\Windows\\system32\\svchost.exe\n'
    '* 672\t532\tlsass.exe\t0xa3066f73e080\t9\t-\t0\tFalse\t2026-03-13 12:58:03.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\lsass.exe\tC:\\Windows\\system32\\lsass.exe\tC:\\Windows\\system32\\lsass.exe\n'
    '540\t524\tcsrss.exe\t0xa3066f6c9140\t13\t-\t1\tFalse\t2026-03-13 12:58:03.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\csrss.exe\t%SystemRoot%\\system32\\csrss.exe ObjectDirectory=\\Windows SharedSection=1024,20480,768 Windows=On SubSystemType=Windows ServerDll=basesrv,1 ServerDll=winsrv:UserServerDllInitialization,3 ServerDll=sxssrv,4 ProfileControl=Off MaxRequestThreads=16\tC:\\Windows\\system32\\csrss.exe\n'
    '548\t524\twinlogon.exe\t0xa3066f6d4080\t5\t-\t1\tFalse\t2026-03-13 12:58:03.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\winlogon.exe\twinlogon.exe\tC:\\Windows\\system32\\winlogon.exe\n'
    '4120\t4088\texplorer.exe\t0xa30670a8c080\t74\t-\t1\tFalse\t2026-03-13 13:04:12.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\explorer.exe\tC:\\Windows\\Explorer.EXE\tC:\\Windows\\Explorer.EXE\n'
    '* 6632\t4120\tOUTLOOK.EXE\t0xa30671c2a080\t48\t-\t1\tFalse\t2026-03-13 13:06:40.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Program Files\\Microsoft Office\\root\\Office16\\OUTLOOK.EXE\t"C:\\Program Files\\Microsoft Office\\root\\Office16\\OUTLOOK.EXE" \tC:\\Program Files\\Microsoft Office\\root\\Office16\\OUTLOOK.EXE\n'
    '** 7316\t6632\tpowershell.exe\t0xa30672d10080\t14\t-\t1\tFalse\t2026-03-13 16:42:18.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\WindowsPowerShell\\v1.0\\powershell.exe\tpowershell.exe -NoProfile -WindowStyle Hidden -EncodedCommand VwByAGkAdABlAC0ATwB1AHQAcAB1AHQAIAAnAEwAQQBCAC0AUAA0ACAAcwB5AG4AdABoAGUAdABpAGMAIABwAGwAYQBjAGUAaABvAGwAZABlAHIALAAgAG4AbwAgAHAAYQB5AGwAbwBhAGQAJwA=\tC:\\Windows\\System32\\WindowsPowerShell\\v1.0\\powershell.exe\n'
    '*** 7488\t7316\tsynchelper.exe\t0xa30672e44080\t9\t-\t1\tFalse\t2026-03-13 16:42:31.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Users\\acct.clerk01\\AppData\\Roaming\\SyncHelper\\synchelper.exe\t"C:\\Users\\acct.clerk01\\AppData\\Roaming\\SyncHelper\\synchelper.exe" --svc\tC:\\Users\\acct.clerk01\\AppData\\Roaming\\SyncHelper\\synchelper.exe\n'
    '**** 7704\t7488\trundll32.exe\t0xa30672f1a080\t5\t-\t1\tFalse\t2026-03-13 16:43:02.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\rundll32.exe\tC:\\Windows\\system32\\rundll32.exe\tC:\\Windows\\system32\\rundll32.exe\n'
    '* 5124\t4120\tsvchost.exe\t0xa30671f03080\t3\t-\t1\tFalse\t2026-03-13 16:44:10.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Users\\Public\\svchost.exe\tC:\\Users\\Public\\svchost.exe\tC:\\Users\\Public\\svchost.exe\n'
    '* 6208\t4120\tmsedge.exe\t0xa30671d7b080\t41\t-\t1\tFalse\t2026-03-13 13:10:55.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Program Files (x86)\\Microsoft\\Edge\\Application\\msedge.exe\t"C:\\Program Files (x86)\\Microsoft\\Edge\\Application\\msedge.exe"\tC:\\Program Files (x86)\\Microsoft\\Edge\\Application\\msedge.exe\n'
))
RESERVA['case-lab-p4/memory/windows.cmdline.txt'] = ''.join((
    '# SYNTHETIC TRAINING DATA (case LAB-P4). Resembles Volatility 3 windows.cmdline output for\n'
    '# fictional host WS-FIN-07. The encoded command decodes to a harmless placeholder string.\n'
    'PID\tProcess\tArgs\n'
    '4120\texplorer.exe\tC:\\Windows\\Explorer.EXE\n'
    '6632\tOUTLOOK.EXE\t"C:\\Program Files\\Microsoft Office\\root\\Office16\\OUTLOOK.EXE"\n'
    '7316\tpowershell.exe\tpowershell.exe -NoProfile -WindowStyle Hidden -EncodedCommand VwByAGkAdABlAC0ATwB1AHQAcAB1AHQAIAAnAEwAQQBCAC0AUAA0ACAAcwB5AG4AdABoAGUAdABpAGMAIABwAGwAYQBjAGUAaABvAGwAZABlAHIALAAgAG4AbwAgAHAAYQB5AGwAbwBhAGQAJwA=\n'
    '7488\tsynchelper.exe\t"C:\\Users\\acct.clerk01\\AppData\\Roaming\\SyncHelper\\synchelper.exe" --svc\n'
    '7704\trundll32.exe\tC:\\Windows\\system32\\rundll32.exe\n'
    '5124\tsvchost.exe\tC:\\Users\\Public\\svchost.exe\n'
    '6208\tmsedge.exe\t"C:\\Program Files (x86)\\Microsoft\\Edge\\Application\\msedge.exe"\n'
))

EVIDENCIA = {}           # o que você vai analisar de fato (nome curto -> texto)
ORIGEM = {}              # de onde veio cada arquivo: "rede" ou "reserva"

for nome, reserva in RESERVA.items():
    texto = None
    if not FORCAR_OFFLINE:
        try:
            # timeout curto: se a rede não responder em 8 s, seguimos com o plano B
            with urllib.request.urlopen(URL_BASE + nome, timeout=8) as resp:
                texto = resp.read().decode("utf-8")
            ORIGEM[nome] = "rede"
        except Exception as erro:            # sem internet, proxy, 404... qualquer falha
            print(f"⚠️  {nome.split('/')[-1]}: rede indisponível ({type(erro).__name__}), usando a cópia embutida")
    if texto is None:
        texto = reserva
        ORIGEM[nome] = "reserva"
    EVIDENCIA[nome.split("/")[-1]] = texto   # guardamos pelo nome curto
    h = hashlib.sha256(texto.encode("utf-8")).hexdigest()
    # Se o upstream mudou depois que este notebook foi gerado, os hashes divergem: avisamos.
    igual = "= cópia embutida" if texto == reserva else "≠ cópia embutida (o upstream mudou!)"
    print(f"{ORIGEM[nome]:8} {h[:16]}…  {nome.split('/')[-1]}  ({igual})")


## 1. Ler o pstree

Linhas que começam com `#` são comentário. O número de `*` no começo é a profundidade. As colunas são separadas por tabulação.


In [ ]:
def ler_pstree(texto):
    """Devolve uma lista de processos (dicionários) a partir da saída tabulada do windows.pstree."""
    linhas = [l for l in texto.splitlines() if l and not l.startswith("#")]
    cab = linhas[0].split("\t")
    procs = []
    for l in linhas[1:]:
        estrelas = len(l) - len(l.lstrip("* "))              # quantos caracteres de "* " no início
        prof = l[:estrelas].count("*")
        campos = l[estrelas:].split("\t")
        p = dict(zip(cab, campos))
        p["depth"] = prof
        p["PID"], p["PPID"] = int(p["PID"]), int(p["PPID"])
        procs.append(p)
    return procs

PROCS = ler_pstree(EVIDENCIA["windows.pstree.txt"])
POR_PID = {p["PID"]: p for p in PROCS}
print(f"{len(PROCS)} processos")
for p in PROCS:
    print(f"{'  ' * p['depth']}{p['ImageFileName']} ({p['PID']})  pai {p['PPID']}  {p['CreateTime'][11:19]}")


## 2. A árvore normal como regras

Cada regra olha **uma coluna** e devolve o motivo. Pai normal e caminho normal são verificados separadamente.


In [ ]:
PAI_ESPERADO = {            # nome -> nomes de pais aceitáveis
    "smss.exe": {"System"}, "services.exe": {"wininit.exe"}, "lsass.exe": {"wininit.exe"},
    "svchost.exe": {"services.exe"}, "OUTLOOK.EXE": {"explorer.exe"}, "msedge.exe": {"explorer.exe"},
}
PAIS_QUE_SAEM = {"wininit.exe", "csrss.exe", "winlogon.exe", "explorer.exe"}   # PPID órfão é normal
FILHOS_ESTRANHOS = {("OUTLOOK.EXE", "powershell.exe"), ("WINWORD.EXE", "powershell.exe"),
                    ("EXCEL.EXE", "powershell.exe")}

def desvios(p, por_pid):
    d = []
    pai = por_pid.get(p["PPID"])
    nome_pai = pai["ImageFileName"] if pai else None
    caminho = p.get("Path", "-")
    if pai is None and p["ImageFileName"] not in PAIS_QUE_SAEM | {"System"}:
        d.append(f"PPID {p['PPID']} não existe")
    if nome_pai and p["ImageFileName"] in PAI_ESPERADO and nome_pai not in PAI_ESPERADO[p["ImageFileName"]]:
        d.append(f"pai {nome_pai} (esperado {', '.join(PAI_ESPERADO[p['ImageFileName']])})")
    if (nome_pai, p["ImageFileName"]) in FILHOS_ESTRANHOS:
        d.append(f"programa de usuário ({nome_pai}) iniciando {p['ImageFileName']}")
    if p["ImageFileName"] == "svchost.exe" and "System32" not in caminho.replace("system32", "System32"):
        d.append(f"svchost fora do System32: {caminho}")
    if "\\AppData\\" in caminho or "\\Users\\Public\\" in caminho:
        d.append(f"executável em pasta de usuário: {caminho}")
    cmd = p.get("Cmd", "")
    if "-EncodedCommand" in cmd or "-WindowStyle Hidden" in cmd:
        d.append("linha de comando com -EncodedCommand / janela oculta")
    # argumento longo que decodifica como Base64 de UTF-16LE legível (o truque do -EncodedCommand, em outro lugar)
    for arg in cmd.split():
        if len(arg) >= 24 and "-EncodedCommand" not in cmd:
            try:
                texto = __import__("base64").b64decode(arg, validate=True).decode("utf-16-le")
                if texto.isprintable():
                    d.append(f"argumento em Base64/UTF-16LE na linha de comando: {arg[:16]}...")
                    break
            except Exception:          # não era Base64, ou não era UTF-16LE: segue o baile
                pass
    if p["ImageFileName"] == "rundll32.exe" and cmd.strip().lower().endswith("rundll32.exe"):
        d.append("rundll32 sem argumento de DLL")
    return d

ACHADOS = {}
for p in PROCS:
    d = desvios(p, POR_PID)
    if d:
        ACHADOS[p["PID"]] = d
        print(f"🔴 {p['ImageFileName']} ({p['PID']}):")
        for x in d:
            print(f"     - {x}")
print(f"\n{len(ACHADOS)} nós vermelhos (o original espera 4)")


> ❓ Por que `wininit.exe` (pai 444) e `csrss.exe` (pai 444 e 524) não acenderam? Porque estão em `PAIS_QUE_SAEM`: os pais são filhos de curta duração do `smss.exe` que já saíram. É o passo 6 do original.

## 3. Decodificar o `-EncodedCommand`

Base64 → bytes → texto UTF-16LE. **Nada é executado**: o resultado é só uma string que vamos documentar.


In [ ]:
import base64

def decodificar_ps(b64):
    return base64.b64decode(b64).decode("utf-16-le", errors="replace")

for linha in EVIDENCIA["windows.cmdline.txt"].splitlines():
    if linha.startswith("7316\t"):                   # o awk -F'\t' '$1==7316' do original
        b64 = linha.split("\t")[2].split()[-1]      # e o awk '{print $NF}'
        print("Base64 :", b64[:40], "...")
        print("Texto  :", decodificar_ps(b64))


## 4. O `pstree.dot`

Gerado a partir dos dados, com **todos** os processos, ramo do sistema no estilo padrão e os vermelhos com o motivo no rótulo.


In [ ]:
import shutil, subprocess
from pathlib import Path

def gerar_dot(procs, achados, nome):
    por_pid = {p["PID"]: p for p in procs}
    out = [f"digraph {nome} {{", '  rankdir=LR; node [shape=box, fontname="Helvetica"];']
    for p in procs:
        rot = f"{p['ImageFileName']} ({p['PID']})"
        if p["PID"] in achados:
            # no .dot, a barra invertida é caractere de escape: dobramos (chr(92) = "\") e trocamos aspas
            motivos = [m.replace(chr(92), chr(92) * 2).replace('"', "'") for m in achados[p["PID"]][:2]]
            motivo = (chr(92) + "n").join(motivos)          # \n = quebra de linha dentro do rótulo
            out.append(f'  p{p["PID"]} [label="{rot}\\n{motivo}", style=filled, fillcolor="#f4b6b6"];')
        else:
            out.append(f'  p{p["PID"]} [label="{rot}"];')
    for p in procs:
        if p["PPID"] in por_pid:
            out.append(f"  p{p['PPID']} -> p{p['PID']};")
        elif p["PPID"] != 0:            # o System (PID 4) tem pai 0: não desenhamos            # pai que saiu: nó pontilhado, como no diagrama do original
            out.append(f'  x{p["PPID"]} [label="PID {p["PPID"]} (saiu)", style=dashed];')
            out.append(f"  x{p['PPID']} -> p{p['PID']} [style=dashed];")
    out.append("}")
    return "\n".join(out)

dot = gerar_dot(PROCS, ACHADOS, "pstree")
Path("pstree.dot").write_text(dot, encoding="utf-8")
print(dot[:900], "\n...")

if shutil.which("dot"):                          # Graphviz instalado? então renderiza
    subprocess.run(["dot", "-Tpng", "pstree.dot", "-o", "ws-fin-07-pstree.png"], check=False)
    print("\n🖼️  ws-fin-07-pstree.png gerado")
else:
    print("\n(Graphviz ausente: cole o .dot em https://dreampuf.github.io/GraphvizOnline ou rode a célula opcional)")


## 5. A lista de achados (passo 5 do original)

Uma linha por nó vermelho, no formato do original. A coluna "explicação legítima" é sua: preencha no formulário.


In [ ]:
#@title Explicações legítimas consideradas { display-mode: "form" }
exp_7316 = "Suplemento do Outlook que chama PowerShell: descartado, a janela oculta e o EncodedCommand não fazem parte de suplementos conhecidos"  #@param {type:"string"}
exp_7488 = "Agente de sincronização instalado pelo usuário: descartado, o $SI foi reescrito (Dia 52)"  #@param {type:"string"}
exp_7704 = "rundll32 iniciado por um painel de controle: descartado, o pai é o synchelper"  #@param {type:"string"}
exp_5124 = "Cópia de svchost por um software de inventário: improvável, pai explorer e pasta Public"  #@param {type:"string"}

EXP = {7316: exp_7316, 7488: exp_7488, 7704: exp_7704, 5124: exp_5124}
linhas = []
for pid, d in ACHADOS.items():
    p = POR_PID[pid]
    linhas.append(f"{p['ImageFileName']} ({pid}) | {'; '.join(d)} | windows.pstree.txt, PID {pid} row | "
                  f"legitimate explanation considered: {EXP.get(pid, '...')}")
nota6 = ("Os PIDs 444 e 524 são instâncias de curta duração do smss.exe (uma por sessão) que iniciaram "
         "csrss/wininit e csrss/winlogon e saíram durante a inicialização. PPID sem processo vivo é normal aqui.")
texto = "\n".join(linhas) + "\n\nPasso 6: " + nota6 + "\n"
Path("ws-fin-07-findings.txt").write_text(texto, encoding="utf-8")
print(texto)


## 6. ➕ O `SA-CX-07`


In [ ]:
# ------------------------------------------------------------
# RECEBER O EVID-SA-003 (MEMÓRIA DO SA-CX-07, TEXTO SINTÉTICO)
# Plano A: baixar do repositório original (GitHub raw), com urllib.
# Plano B: usar a cópia embutida logo abaixo (o arquivo inteiro,
#          copiado do upstream/), para o notebook rodar sem internet.
# ------------------------------------------------------------
import urllib.request      # baixa arquivos da web, sem instalar nada
import hashlib             # SHA-256: a impressão digital dos bytes

FORCAR_OFFLINE = False     # True = nem tenta a rede (útil em sala sem Wi-Fi)
URL_BASE = 'https://raw.githubusercontent.com/Ridd1kulusC0d3r/investigador-cyber-90-dias/main/caso-continuo/dados/'

RESERVA = {}             # caminho do arquivo -> conteúdo embutido (plano B)
RESERVA['dia-53/sa-cx-07.windows.pstree.txt'] = ''.join((
    '# DADOS SINTÉTICOS (Operação Isca). Texto escrito à mão no layout do Volatility 3 para o host\n'
    '# fictício SA-CX-07 (10.20.40.57). NÃO existe imagem de memória por trás. Separado por tabulação.\n'
    'PID\tPPID\tImageFileName\tOffset(V)\tThreads\tHandles\tSessionId\tWow64\tCreateTime\tExitTime\tAudit\tCmd\tPath\n'
    '4\t0\tSystem\t0xb2051c600080\t148\t-\tN/A\tFalse\t2026-10-05 10:50:41.000000 UTC\tN/A\t-\t-\t-\n'
    '* 96\t4\tRegistry\t0xb2051c641080\t4\t-\tN/A\tFalse\t2026-10-05 10:50:38.000000 UTC\tN/A\tRegistry\t-\t-\n'
    '* 380\t4\tsmss.exe\t0xb2051c682080\t2\t-\tN/A\tFalse\t2026-10-05 10:50:41.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\smss.exe\t\\SystemRoot\\System32\\smss.exe\t\\SystemRoot\\System32\\smss.exe\n'
    '492\t480\tcsrss.exe\t0xb2051c6c3080\t11\t-\t0\tFalse\t2026-10-05 10:50:43.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\csrss.exe\t%SystemRoot%\\system32\\csrss.exe ObjectDirectory=\\Windows\tC:\\Windows\\system32\\csrss.exe\n'
    '568\t480\twininit.exe\t0xb2051c704080\t1\t-\t0\tFalse\t2026-10-05 10:50:43.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\wininit.exe\twininit.exe\tC:\\Windows\\system32\\wininit.exe\n'
    '* 700\t568\tservices.exe\t0xb2051c745080\t6\t-\t0\tFalse\t2026-10-05 10:50:43.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\services.exe\tC:\\Windows\\system32\\services.exe\tC:\\Windows\\system32\\services.exe\n'
    '** 812\t700\tsvchost.exe\t0xb2051c786080\t20\t-\t0\tFalse\t2026-10-05 10:50:44.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\svchost.exe\tC:\\Windows\\system32\\svchost.exe -k DcomLaunch -p\tC:\\Windows\\system32\\svchost.exe\n'
    '** 868\t700\tsvchost.exe\t0xb2051c7c7080\t12\t-\t0\tFalse\t2026-10-05 10:50:44.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\svchost.exe\tC:\\Windows\\system32\\svchost.exe -k RPCSS -p\tC:\\Windows\\system32\\svchost.exe\n'
    '** 1204\t700\tsvchost.exe\t0xb2051c808080\t36\t-\t0\tFalse\t2026-10-05 10:50:45.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\svchost.exe\tC:\\Windows\\system32\\svchost.exe -k netsvcs -p\tC:\\Windows\\system32\\svchost.exe\n'
    '* 716\t568\tlsass.exe\t0xb2051c849080\t9\t-\t0\tFalse\t2026-10-05 10:50:43.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\lsass.exe\tC:\\Windows\\system32\\lsass.exe\tC:\\Windows\\system32\\lsass.exe\n'
    '580\t560\tcsrss.exe\t0xb2051c88a080\t13\t-\t1\tFalse\t2026-10-05 10:50:43.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\csrss.exe\t%SystemRoot%\\system32\\csrss.exe ObjectDirectory=\\Windows\tC:\\Windows\\system32\\csrss.exe\n'
    '612\t560\twinlogon.exe\t0xb2051c8cb080\t5\t-\t1\tFalse\t2026-10-05 10:50:43.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\System32\\winlogon.exe\twinlogon.exe\tC:\\Windows\\system32\\winlogon.exe\n'
    '3988\t3940\texplorer.exe\t0xb2051c90c080\t70\t-\t1\tFalse\t2026-10-05 10:52:31.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Windows\\explorer.exe\tC:\\Windows\\Explorer.EXE\tC:\\Windows\\Explorer.EXE\n'
    '* 5216\t3988\tmsedge.exe\t0xb2051c94d080\t40\t-\t1\tFalse\t2026-10-05 10:53:40.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Program Files (x86)\\Microsoft\\Edge\\Application\\msedge.exe\t"C:\\Program Files (x86)\\Microsoft\\Edge\\Application\\msedge.exe"\tC:\\Program Files (x86)\\Microsoft\\Edge\\Application\\msedge.exe\n'
    '* 6120\t3988\tcaixa-sa.exe\t0xb2051c98e080\t22\t-\t1\tFalse\t2026-10-05 10:55:01.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Program Files\\SerraAzul\\Caixa\\caixa-sa.exe\t"C:\\Program Files\\SerraAzul\\Caixa\\caixa-sa.exe" --agencia 0001\tC:\\Program Files\\SerraAzul\\Caixa\\caixa-sa.exe\n'
    '* 7712\t3988\tmsa-atualizado\t0xb2051c9cf080\t6\t-\t1\tFalse\t2026-10-05 11:24:31.000000 UTC\tN/A\t\\Device\\HarddiskVolume3\\Users\\cx.centro07\\AppData\\Roaming\\MSA\\msa-atualizador.exe\t"C:\\Users\\cx.centro07\\AppData\\Roaming\\MSA\\msa-atualizador.exe" --canal YQB0AHUAYQBsAGkAegBhAC0AbQBvAGQAdQBsAG8ALgBlAHgAYQBtAHAAbABlAA==\tC:\\Users\\cx.centro07\\AppData\\Roaming\\MSA\\msa-atualizador.exe\n'
))
RESERVA['dia-53/sa-cx-07.windows.cmdline.txt'] = ''.join((
    '# DADOS SINTÉTICOS (Operação Isca). Texto escrito à mão no layout do Volatility 3 para o host\n'
    '# fictício SA-CX-07 (10.20.40.57). NÃO existe imagem de memória por trás. Separado por tabulação.\n'
    'PID\tProcess\tArgs\n'
    '3988\texplorer.exe\tC:\\Windows\\Explorer.EXE\n'
    '5216\tmsedge.exe\t"C:\\Program Files (x86)\\Microsoft\\Edge\\Application\\msedge.exe"\n'
    '6120\tcaixa-sa.exe\t"C:\\Program Files\\SerraAzul\\Caixa\\caixa-sa.exe" --agencia 0001\n'
    '7712\tmsa-atualizado\t"C:\\Users\\cx.centro07\\AppData\\Roaming\\MSA\\msa-atualizador.exe" --canal YQB0AHUAYQBsAGkAegBhAC0AbQBvAGQAdQBsAG8ALgBlAHgAYQBtAHAAbABlAA==\n'
))
RESERVA['dia-53/sa-cx-07.windows.pslist.txt'] = ''.join((
    '# DADOS SINTÉTICOS (Operação Isca). Texto escrito à mão no layout do Volatility 3 para o host\n'
    '# fictício SA-CX-07 (10.20.40.57). NÃO existe imagem de memória por trás. Separado por tabulação.\n'
    'PID\tPPID\tImageFileName\tOffset(V)\tThreads\tHandles\tSessionId\tWow64\tCreateTime\tExitTime\n'
    '4\t0\tSystem\t0xb2051c600080\t148\t-\tN/A\tFalse\t2026-10-05 10:50:41.000000 UTC\tN/A\n'
    '96\t4\tRegistry\t0xb2051c641080\t4\t-\tN/A\tFalse\t2026-10-05 10:50:38.000000 UTC\tN/A\n'
    '380\t4\tsmss.exe\t0xb2051c682080\t2\t-\tN/A\tFalse\t2026-10-05 10:50:41.000000 UTC\tN/A\n'
    '492\t480\tcsrss.exe\t0xb2051c6c3080\t11\t-\t0\tFalse\t2026-10-05 10:50:43.000000 UTC\tN/A\n'
    '568\t480\twininit.exe\t0xb2051c704080\t1\t-\t0\tFalse\t2026-10-05 10:50:43.000000 UTC\tN/A\n'
    '700\t568\tservices.exe\t0xb2051c745080\t6\t-\t0\tFalse\t2026-10-05 10:50:43.000000 UTC\tN/A\n'
    '812\t700\tsvchost.exe\t0xb2051c786080\t20\t-\t0\tFalse\t2026-10-05 10:50:44.000000 UTC\tN/A\n'
    '868\t700\tsvchost.exe\t0xb2051c7c7080\t12\t-\t0\tFalse\t2026-10-05 10:50:44.000000 UTC\tN/A\n'
    '1204\t700\tsvchost.exe\t0xb2051c808080\t36\t-\t0\tFalse\t2026-10-05 10:50:45.000000 UTC\tN/A\n'
    '716\t568\tlsass.exe\t0xb2051c849080\t9\t-\t0\tFalse\t2026-10-05 10:50:43.000000 UTC\tN/A\n'
    '580\t560\tcsrss.exe\t0xb2051c88a080\t13\t-\t1\tFalse\t2026-10-05 10:50:43.000000 UTC\tN/A\n'
    '612\t560\twinlogon.exe\t0xb2051c8cb080\t5\t-\t1\tFalse\t2026-10-05 10:50:43.000000 UTC\tN/A\n'
    '3988\t3940\texplorer.exe\t0xb2051c90c080\t70\t-\t1\tFalse\t2026-10-05 10:52:31.000000 UTC\tN/A\n'
    '5216\t3988\tmsedge.exe\t0xb2051c94d080\t40\t-\t1\tFalse\t2026-10-05 10:53:40.000000 UTC\tN/A\n'
    '6120\t3988\tcaixa-sa.exe\t0xb2051c98e080\t22\t-\t1\tFalse\t2026-10-05 10:55:01.000000 UTC\tN/A\n'
    '7712\t3988\tmsa-atualizado\t0xb2051c9cf080\t6\t-\t1\tFalse\t2026-10-05 11:24:31.000000 UTC\tN/A\n'
))
RESERVA['dia-53/sa-cx-07.windows.psscan.txt'] = ''.join((
    '# DADOS SINTÉTICOS (Operação Isca). Texto escrito à mão no layout do Volatility 3 para o host\n'
    '# fictício SA-CX-07 (10.20.40.57). NÃO existe imagem de memória por trás. Separado por tabulação.\n'
    'PID\tPPID\tImageFileName\tOffset(V)\tThreads\tHandles\tSessionId\tWow64\tCreateTime\tExitTime\n'
    '4\t0\tSystem\t0xb2051c600080\t148\t-\tN/A\tFalse\t2026-10-05 10:50:41.000000 UTC\tN/A\n'
    '96\t4\tRegistry\t0xb2051c641080\t4\t-\tN/A\tFalse\t2026-10-05 10:50:38.000000 UTC\tN/A\n'
    '380\t4\tsmss.exe\t0xb2051c682080\t2\t-\tN/A\tFalse\t2026-10-05 10:50:41.000000 UTC\tN/A\n'
    '492\t480\tcsrss.exe\t0xb2051c6c3080\t11\t-\t0\tFalse\t2026-10-05 10:50:43.000000 UTC\tN/A\n'
    '568\t480\twininit.exe\t0xb2051c704080\t1\t-\t0\tFalse\t2026-10-05 10:50:43.000000 UTC\tN/A\n'
    '700\t568\tservices.exe\t0xb2051c745080\t6\t-\t0\tFalse\t2026-10-05 10:50:43.000000 UTC\tN/A\n'
    '812\t700\tsvchost.exe\t0xb2051c786080\t20\t-\t0\tFalse\t2026-10-05 10:50:44.000000 UTC\tN/A\n'
    '868\t700\tsvchost.exe\t0xb2051c7c7080\t12\t-\t0\tFalse\t2026-10-05 10:50:44.000000 UTC\tN/A\n'
    '1204\t700\tsvchost.exe\t0xb2051c808080\t36\t-\t0\tFalse\t2026-10-05 10:50:45.000000 UTC\tN/A\n'
    '716\t568\tlsass.exe\t0xb2051c849080\t9\t-\t0\tFalse\t2026-10-05 10:50:43.000000 UTC\tN/A\n'
    '580\t560\tcsrss.exe\t0xb2051c88a080\t13\t-\t1\tFalse\t2026-10-05 10:50:43.000000 UTC\tN/A\n'
    '612\t560\twinlogon.exe\t0xb2051c8cb080\t5\t-\t1\tFalse\t2026-10-05 10:50:43.000000 UTC\tN/A\n'
    '3988\t3940\texplorer.exe\t0xb2051c90c080\t70\t-\t1\tFalse\t2026-10-05 10:52:31.000000 UTC\tN/A\n'
    '5216\t3988\tmsedge.exe\t0xb2051c94d080\t40\t-\t1\tFalse\t2026-10-05 10:53:40.000000 UTC\tN/A\n'
    '6120\t3988\tcaixa-sa.exe\t0xb2051c98e080\t22\t-\t1\tFalse\t2026-10-05 10:55:01.000000 UTC\tN/A\n'
    '7712\t3988\tmsa-atualizado\t0xb2051c9cf080\t6\t-\t1\tFalse\t2026-10-05 11:24:31.000000 UTC\tN/A\n'
    '7790\t7712\tschtasks.exe\t0xb2051f8e1080\t0\t-\t1\tFalse\t2026-10-05 11:24:44.000000 UTC\t2026-10-05 11:24:45.000000 UTC\n'
))

MEM_SA = {}           # o que você vai analisar de fato (nome curto -> texto)
ORIGEM = {}              # de onde veio cada arquivo: "rede" ou "reserva"

for nome, reserva in RESERVA.items():
    texto = None
    if not FORCAR_OFFLINE:
        try:
            # timeout curto: se a rede não responder em 8 s, seguimos com o plano B
            with urllib.request.urlopen(URL_BASE + nome, timeout=8) as resp:
                texto = resp.read().decode("utf-8")
            ORIGEM[nome] = "rede"
        except Exception as erro:            # sem internet, proxy, 404... qualquer falha
            print(f"⚠️  {nome.split('/')[-1]}: rede indisponível ({type(erro).__name__}), usando a cópia embutida")
    if texto is None:
        texto = reserva
        ORIGEM[nome] = "reserva"
    MEM_SA[nome.split("/")[-1]] = texto   # guardamos pelo nome curto
    h = hashlib.sha256(texto.encode("utf-8")).hexdigest()
    # Se o upstream mudou depois que este notebook foi gerado, os hashes divergem: avisamos.
    igual = "= cópia embutida" if texto == reserva else "≠ cópia embutida (o upstream mudou!)"
    print(f"{ORIGEM[nome]:8} {h[:16]}…  {nome.split('/')[-1]}  ({igual})")


In [ ]:
PROCS_SA = ler_pstree(MEM_SA["sa-cx-07.windows.pstree.txt"])
POR_PID_SA = {p["PID"]: p for p in PROCS_SA}
ACH_SA = {}
for p in PROCS_SA:
    d = desvios(p, POR_PID_SA)
    pai = POR_PID_SA.get(p["PPID"], {}).get("ImageFileName", f"PID {p['PPID']} (saiu)" if p["PPID"] else "-")
    marca = "🔴" if d else "  "
    print(f"{marca} {p['ImageFileName']:15} ({p['PID']:>5}) pai {pai:13} {p['CreateTime'][11:19]}  {'; '.join(d)}")
    if d:
        ACH_SA[p["PID"]] = d

print("\nO controle: caixa-sa.exe, filho do explorer, em Program Files\\SerraAzul: nenhuma regra acende.")
print("Repare: o msa-atualizado (7712) tem pai NORMAL. Os desvios estão no caminho e na linha de comando.")


In [ ]:
# O --canal: Base64 de UTF-16LE, como o -EncodedCommand
for linha in MEM_SA["sa-cx-07.windows.cmdline.txt"].splitlines():
    if linha.startswith("7712\t"):
        b64 = linha.split("--canal ")[1].split()[0]
        print("--canal decodificado:", decodificar_ps(b64))
print("Já visto no proxy (Dia 49): CONNECT atualiza-modulo.example:443 a partir de 10.20.40.57.")
print("Ligação a escrever: MESMO NOME DE DOMÍNIO. Não: 'o programa se comunicou com'. Isso é o Dia 54.")


### pslist × psscan (o `comm -13` do Exercício 3)


In [ ]:
def pids(texto):
    """Conjunto (PID, nome) de uma saída tabulada do pslist/psscan."""
    s = set()
    for l in texto.splitlines()[1:]:
        if l and not l.startswith("#") and l.split("\t")[0].isdigit():
            c = l.split("\t")
            s.add((int(c[0]), c[2], int(c[1]), c[9]))
    return s

so_no_psscan = pids(MEM_SA["sa-cx-07.windows.psscan.txt"]) - pids(MEM_SA["sa-cx-07.windows.pslist.txt"])
for pid, nome, ppid, saida in so_no_psscan:
    print(f"Só no psscan: {nome} ({pid}), pai {ppid} ({POR_PID_SA[ppid]['ImageFileName']}), ExitTime {saida}")
print("\nFonte independente: 4688 do schtasks.exe às 11:24:44 UTC, pai msa-atualizador.exe (log Security, pacote do Dia 57).")
print("O que o psscan NÃO diz: o nome da tarefa criada. Isso vem do disco (C:\\Windows\\System32\\Tasks) ou do registro.")

dot_sa = gerar_dot(PROCS_SA, ACH_SA, "pstree_sa")
Path("sa-cx-07-pstree.dot").write_text(dot_sa, encoding="utf-8")
print("\n💾 sa-cx-07-pstree.dot")


## 7. 🎮 Quiz do pai certo


In [ ]:
MODO_JOGO = True
QUIZ = [
    ("services.exe", ["explorer.exe", "wininit.exe", "smss.exe"], 2),
    ("lsass.exe", ["wininit.exe", "services.exe", "svchost.exe"], 1),
    ("svchost.exe legítimo", ["explorer.exe", "services.exe", "winlogon.exe"], 2),
    ("smss.exe", ["System", "wininit.exe", "csrss.exe"], 1),
    ("OUTLOOK.EXE", ["services.exe", "svchost.exe", "explorer.exe"], 3),
    ("explorer.exe (o PPID que aparece é de um processo que...)", ["está oculto", "saiu (userinit.exe)", "é malware"], 2),
    ("Plugin para achar um processo que JÁ SAIU", ["windows.pslist", "windows.psscan", "windows.cmdline"], 2),
]
acertos = 0
for i, (proc, ops, certa) in enumerate(QUIZ, 1):
    print(f"\n[{i}] Pai normal de: {proc}")
    for k, o in enumerate(ops, 1):
        print(f"   {k}. {o}")
    if MODO_JOGO:
        ok = input("   Resposta: ").strip() == str(certa)
        acertos += ok
        print("   ✅" if ok else f"   ❌ certa: {ops[certa - 1]}")
    else:
        print(f"   certa: {ops[certa - 1]}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}")


In [ ]:
try:
    from google.colab import files
    for f in ("pstree.dot", "ws-fin-07-findings.txt", "sa-cx-07-pstree.dot"):
        files.download(f)
except ImportError:
    print("(Fora do Colab: pstree.dot, ws-fin-07-findings.txt e sa-cx-07-pstree.dot na pasta atual)")


## ✅ Antes de fechar o Dia 53

- [ ] grafo com os 19 processos do WS-FIN-07, quatro vermelhos com motivo por coluna
- [ ] lista de achados com explicação legítima considerada
- [ ] nota sobre os PIDs 444 e 524
- [ ] ➕ `SA-CX-07`: 7712 com pai normal e três desvios; `caixa-sa.exe` como controle; `--canal` decodificado
- [ ] ➕ `schtasks.exe` só no psscan, com o 4688 como segunda fonte

**Amanhã (Dia 54):** o que esses processos estavam fazendo: conexões de rede (`netscan`) e código injetado (`malfind`).

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>
